# 06 - Graduated Compression Curve

Generate five compression levels of each COT (light, medium, heavy, ultra-heavy, single-sentence),
then prefill PRIMARY_MODEL with each. Plot accuracy as a function of compression level.

The *shape* of this curve characterises legibility: a sharp cliff means a phase transition;
a gentle slope means information is distributed continuously across surface features.

In [3]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Already up to date.


In [4]:
import json
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from lib.prompts import COMPRESSION_LEVELS, build_compression_messages
from lib.data import extract_predicted_answer
from lib.prefill import run_prefill_condition

# Load all COTs from the core run
cots = []
for p in sorted(COT_CACHE.glob("*.json")):
    cots.append(json.loads(p.read_text()))
print(f"Loaded {len(cots)} COTs")

LEVELS = ["light", "medium", "heavy", "ultra_heavy", "single_sentence"]

ImportError: cannot import name 'COMPRESSION_LEVELS' from 'lib.prompts' (/workspace/13-4-2026/legibility/lib/prompts.py)

## Phase 1: Generate compressed versions at all levels

Light and heavy already exist from 03_paraphrase. Generate medium, ultra_heavy, single_sentence.

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# Only need to generate new levels
new_levels = ["medium", "ultra_heavy", "single_sentence"]

# Check what's already done
for level in LEVELS:
    condition = f"compress_{level}"
    n = len(list(PARAPHRASE_CACHE.glob(f"{condition}_*.json")))
    print(f"{condition}: {n} cached")

# Also count existing paraphrase_light and paraphrase_heavy
n_light = len(list(PARAPHRASE_CACHE.glob("paraphrase_light_*.json")))
n_heavy = len(list(PARAPHRASE_CACHE.glob("paraphrase_heavy_*.json")))
print(f"\nExisting: paraphrase_light={n_light}, paraphrase_heavy={n_heavy}")

In [ ]:
print(f"Loading {PARAPHRASER_MODEL}...")
llm = LLM(model=PARAPHRASER_MODEL, dtype="bfloat16", max_model_len=4096)
tokenizer = AutoTokenizer.from_pretrained(PARAPHRASER_MODEL)
print("Paraphraser loaded.")

In [ ]:
sampling = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_COT_TOKENS)

for level in new_levels:
    condition = f"compress_{level}"
    done_ids = {
        int(p.stem.split("_", 2)[-1])
        for p in PARAPHRASE_CACHE.glob(f"{condition}_*.json")
    }
    todo = [c for c in cots if c["problem_id"] not in done_ids]
    print(f"[{condition}] {len(done_ids)} done, {len(todo)} remaining")

    for i in tqdm(range(0, len(todo), CHUNK_SIZE), desc=condition):
        chunk = todo[i:i + CHUNK_SIZE]
        messages_batch = [build_compression_messages(c["cot_text"], level) for c in chunk]
        prompts = [
            tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            for msgs in messages_batch
        ]
        outputs = llm.generate(prompts, sampling)

        for c, output in zip(chunk, outputs):
            result = {
                "problem_id": c["problem_id"],
                "condition": condition,
                "original_cot": c["cot_text"],
                "paraphrased_cot": output.outputs[0].text.strip(),
            }
            path = PARAPHRASE_CACHE / f"{condition}_{c['problem_id']}.json"
            path.write_text(json.dumps(result))

print("All compression levels generated.")

In [ ]:
del llm
import gc; gc.collect()
import torch; torch.cuda.empty_cache()

## Phase 2: Prefill PRIMARY_MODEL with each compression level

In [ ]:
from lib.data import load_gsm8k
dataset = load_gsm8k()

# Build lookup for each compression level
def load_compression_lookup(level):
    # For light/heavy, reuse existing paraphrase cache
    if level == "light":
        prefix = "paraphrase_light"
    elif level == "heavy":
        prefix = "paraphrase_heavy"
    else:
        prefix = f"compress_{level}"

    lookup = {}
    for p in PARAPHRASE_CACHE.glob(f"{prefix}_*.json"):
        r = json.loads(p.read_text())
        lookup[r["problem_id"]] = r["paraphrased_cot"]
    return lookup

compression_lookups = {level: load_compression_lookup(level) for level in LEVELS}
for level, lookup in compression_lookups.items():
    print(f"{level}: {len(lookup)} entries")

In [ ]:
print(f"Loading {PRIMARY_MODEL}...")
llm = LLM(model=PRIMARY_MODEL, dtype="bfloat16", max_model_len=4096)
print("PRIMARY_MODEL loaded.")

for level in LEVELS:
    condition = f"compress_{level}_self"
    run_prefill_condition(llm, condition, PRIMARY_MODEL, dataset, compression_lookups[level])

del llm
import gc; gc.collect()
import torch; torch.cuda.empty_cache()

## Analysis: Compression curve

In [ ]:
# Compute accuracy at each compression level
level_labels = ["Verbatim", "Light", "Medium", "Heavy", "Ultra-heavy", "Single sent."]
level_conditions = ["self_prefill"] + [f"compress_{l}_self" for l in LEVELS]

accs = []
cis = []

def bootstrap_ci(arr, n=10000, seed=42):
    rng = np.random.RandomState(seed)
    boots = [rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n)]
    boots = np.sort(boots)
    return boots[int(0.025 * n)], boots[int(0.975 * n)]

for cond in level_conditions:
    results = []
    for p in PREFILL_CACHE.glob(f"{cond}_*.json"):
        r = json.loads(p.read_text())
        results.append(r["predicted_answer"] == r["gold_answer"])
    if results:
        arr = np.array(results, dtype=float)
        acc = arr.mean()
        lo, hi = bootstrap_ci(arr)
        accs.append(acc)
        cis.append([acc - lo, hi - acc])
    else:
        accs.append(np.nan)
        cis.append([0, 0])
    print(f"{cond:30s}: {accs[-1]:.3f}")

# No-COT baseline
nc_results = []
for p in PREFILL_CACHE.glob("no_cot_*.json"):
    r = json.loads(p.read_text())
    nc_results.append(r["predicted_answer"] == r["gold_answer"])
no_cot_acc = np.mean(nc_results) if nc_results else 0

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ci_arr = np.array(cis).T
valid = [i for i, a in enumerate(accs) if not np.isnan(a)]

ax.errorbar(
    [i for i in valid], [accs[i] for i in valid],
    yerr=[[ci_arr[0][i] for i in valid], [ci_arr[1][i] for i in valid]],
    marker="o", capsize=4, linewidth=2, markersize=8, color="#2980b9",
)
ax.axhline(y=no_cot_acc, color="gray", linestyle="--", alpha=0.5, label=f"No-COT baseline ({no_cot_acc:.3f})")

ax.set_xticks(range(len(level_labels)))
ax.set_xticklabels(level_labels, rotation=15)
ax.set_ylabel("Accuracy")
ax.set_xlabel("Compression level")
ax.set_title("Graduated Compression Curve", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1)
ax.legend()

for i in valid:
    ax.annotate(f"{accs[i]:.3f}", (i, accs[i]), textcoords="offset points",
                xytext=(0, 12), ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "compression_curve.png", dpi=200, bbox_inches="tight")
plt.show()